# MCBench Demo Results — Simulated Leaderboard

This notebook shows **what MCBench results look like** using simulated model responses
that match expected real-world performance levels.

Simulation parameters are grounded in published benchmarks:
- Frontier models: ~85% MMLU-Pro accuracy, well-calibrated confidence
- Mid-tier models: ~65-70% accuracy, moderate calibration
- Small models: ~55% accuracy, often overconfident

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from src.metrics.mci import compute_mci, build_leaderboard

np.random.seed(2026)
N = 300   # items per model (eval subset)

# ── Simulate 7 representative models ─────────────────────────────────────────
MODEL_SPECS = [
    # name,                  acc,   meta,        const_conf
    ('claude-3-5-sonnet',    0.87,  'excellent',  None),
    ('gpt-4o',               0.85,  'excellent',  None),
    ('gemini-1-5-pro',       0.82,  'good',       None),
    ('claude-3-haiku',       0.71,  'moderate',   None),
    ('gpt-3-5-turbo',        0.63,  'poor',       None),
    ('gemini-flash-1-5',     0.69,  'moderate',   None),
    ('llama-3-8b',           0.55,  'constant',   95),   # always ~95% confident
]

SRS_SCORES = {
    'claude-3-5-sonnet': 0.68,
    'gpt-4o':            0.64,
    'gemini-1-5-pro':    0.59,
    'claude-3-haiku':    0.45,
    'gpt-3-5-turbo':     0.38,
    'gemini-flash-1-5':  0.42,
    'llama-3-8b':        0.34,   # near-chance
}

def simulate_model(acc_rate, meta_quality, const_conf=None, n=N):
    acc = np.random.binomial(1, acc_rate, n)
    if const_conf is not None:
        conf = np.full(n, float(const_conf))
    elif meta_quality == 'excellent':
        conf = np.where(acc == 1,
            np.random.normal(82, 8, n).clip(55, 100),
            np.random.normal(42, 12, n).clip(5, 70))
    elif meta_quality == 'good':
        conf = np.where(acc == 1,
            np.random.normal(78, 10, n).clip(50, 100),
            np.random.normal(48, 14, n).clip(10, 80))
    elif meta_quality == 'moderate':
        conf = np.where(acc == 1,
            np.random.normal(72, 12, n).clip(40, 100),
            np.random.normal(58, 15, n).clip(15, 90))
    else:  # poor
        conf = np.random.uniform(45, 75, n)
    return acc.tolist(), conf.tolist()


results = {}
for name, acc, meta, const_conf in MODEL_SPECS:
    accuracy, confidence = simulate_model(acc, meta, const_conf)
    srs = SRS_SCORES[name]
    results[name] = compute_mci(accuracy, confidence, srs)

leaderboard = build_leaderboard(results)
print('Simulation complete.')

In [ ]:
# ── Display leaderboard ───────────────────────────────────────────────────────
display_cols = ['model', 'mci', 'm_ratio', 'd_prime', 'type2_auroc', 'srs', 'ece', 'reliable']
display(leaderboard[display_cols].style
    .format({'mci': '{:.4f}', 'm_ratio': '{:.4f}', 'd_prime': '{:.4f}',
             'type2_auroc': '{:.4f}', 'srs': '{:.4f}', 'ece': '{:.4f}'})
    .background_gradient(subset=['mci'], cmap='YlOrRd')
    .set_caption('MCBench Simulated Rankings — Frontier > Mid-tier > Small models')
)

In [ ]:
# ── Publication-quality figure ────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.suptitle('MCBench: Metacognitive Capability Across Model Tiers', fontsize=14, fontweight='bold')

lb = leaderboard.sort_values('mci', ascending=True).copy()

# Colour by model tier
frontier = {'claude-3-5-sonnet', 'gpt-4o', 'gemini-1-5-pro'}
mid = {'claude-3-haiku', 'gemini-flash-1-5'}
colors = [
    '#d62728' if m in frontier else '#1f77b4' if m in mid else '#7f7f7f'
    for m in lb['model']
]

# Panel 1: MCI scores
bars = axes[0].barh(lb['model'], lb['mci'], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('MCI Score')
axes[0].set_title('Metacognitive Capability Index')
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, lb['mci']):
    axes[0].text(max(val + 0.01, 0.01), bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=8)

# Panel 2: M-ratio (metacognitive efficiency)
m_ratios = lb['m_ratio'].clip(lower=0)
bars2 = axes[1].barh(lb['model'], m_ratios, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('M-ratio (meta-d\'/d\')  ↑ better')
axes[1].set_title('Metacognitive Efficiency\n(M-ratio = 1 → ideal)')
axes[1].axvline(1.0, color='green', linewidth=1.2, linestyle='--', alpha=0.7, label='ideal M=1')
axes[1].legend(fontsize=8, loc='lower right')

# Panel 3: SRS vs ECE scatter
for i, row in lb.iterrows():
    m = row['model']
    c = '#d62728' if m in frontier else '#1f77b4' if m in mid else '#7f7f7f'
    axes[2].scatter(row['srs'], row['ece'], s=100, color=c, zorder=5)
    axes[2].annotate(m.replace('-', '\n'), (row['srs'], row['ece']),
                     fontsize=7, xytext=(4, 3), textcoords='offset points')

axes[2].set_xlabel('SRS (Self-Recognition Score)  →')
axes[2].set_ylabel('ECE (Calibration Error)  ↓')
axes[2].set_title('Attribution vs Calibration\n(lower-right is better)')

# Legend
patches = [
    mpatches.Patch(color='#d62728', label='Frontier'),
    mpatches.Patch(color='#1f77b4', label='Mid-tier'),
    mpatches.Patch(color='#7f7f7f', label='Small / Constant'),
]
fig.legend(handles=patches, loc='upper right', bbox_to_anchor=(0.99, 0.98), fontsize=9)

plt.tight_layout(rect=[0, 0, 0.96, 1])
plt.savefig('mcbench_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mcbench_results.png')

In [ ]:
# ── Print key findings ────────────────────────────────────────────────────────
top3 = leaderboard.head(3)
bot2 = leaderboard.tail(2)

print('KEY FINDINGS')
print('=' * 50)
print(f'Top-3 models (frontier): {list(top3["model"])}')
print(f'  Mean MCI: {top3["mci"].mean():.4f}')
print(f'  Mean M-ratio: {top3["m_ratio"].mean():.4f}')
print()
print(f'Bottom-2 models (small/constant): {list(bot2["model"])}')
print(f'  Mean MCI: {bot2["mci"].mean():.4f}')
print()
mci_range = leaderboard['mci'].max() - leaderboard['mci'].min()
print(f'MCI range: {mci_range:.4f} (separation across tiers)')

# Constant-confidence model
llama = leaderboard[leaderboard['model'] == 'llama-3-8b'].iloc[0]
print(f'\nConstant-confidence model (llama-3-8b):')
print(f'  gamma = {llama["gamma"]:.4f} (max dynamic penalty = 0.30)')
print(f'  ECE = {llama["ece"]:.4f}')
print(f'  MCI = {llama["mci"]:.4f} (appropriately penalised)')